# 01 - Reanálise reproduzível
Executa a auditoria CPU, os testes e os gráficos. Não treina um LLM. Na primeira execução, mantenha INSTALL_DEPS = True. A instalação e os cálculos usam processos separados; não é necessário reiniciar para executar esta reanálise. Logs completos ficam em logs/.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "revisao_2026", cwd.parent / "revisao_2026",
              cwd / "cot-obfuscation-h2a" / "revisao_2026"]
ROOT = next((p for p in candidates if (p / "src/instruments.py").is_file()), None)
if ROOT is None:
    clone = cwd / "cot-obfuscation-h2a"
    if not clone.exists():
        subprocess.run(["git", "clone", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git",
                        str(clone)], check=True)
    ROOT = clone / "revisao_2026"
assert (ROOT / "src/instruments.py").is_file(), ROOT
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
ENV = dict(os.environ, PYTHONPATH=str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""))
from notebook_runtime import run_logged

def run_step(command, log_name):
    # Atualiza credenciais/variáveis configuradas depois da célula inicial.
    current_env = dict(os.environ)
    current_env["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
    current_env["PYTHONUNBUFFERED"] = "1"
    return run_logged(command, ROOT / "logs" / log_name, cwd=ROOT, env=current_env)

print("Raiz:", ROOT)


In [ ]:
INSTALL_DEPS = True  # Instala as versões fixadas; não exige GPU.
if INSTALL_DEPS:
    run_step([sys.executable, "-m", "pip", "install", "-r", "requirements.lock.txt"], "01_install.log")
run_step([sys.executable, "-u", "-m", "unittest", "discover", "-s", "tests", "-v"], "01_tests.log")
run_step([sys.executable, "-u", "src/reanalyse.py"], "01_reanalysis.log")


In [ ]:
# Biblioteca padrão: evita importar no kernel pacotes já carregados
# antes da instalação. Os cálculos usam o ambiente recém-instalado.
import csv, html
from IPython.display import display, HTML
for filename in ["summary.csv", "contrasts.csv", "grading_stress_summary.csv"]:
    with (ROOT / "results" / filename).open(encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))
    header = "".join("<th>" + html.escape(v) + "</th>" for v in rows[0])
    body = "".join("<tr>" + "".join("<td>" + html.escape(v) + "</td>" for v in row) + "</tr>" for row in rows[1:])
    display(HTML("<h3>" + filename + "</h3><table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))


Os resultados são reprocessamento de métricas depositadas e testes mecânicos. As 540 saídas integrais não estão no CSV original.
